In [4]:
import pandas as pd
from evidently import Dataset
from evidently import DataDefinition
from evidently import Report
from evidently.presets import TextEvals
from evidently.tests import lte, gte, eq
from evidently.descriptors import LLMEval, TestSummary, DeclineLLMEval, Sentiment, TextLength, IncludesWords
from evidently.llm.templates import BinaryClassificationPromptTemplate
from evidently.ui.workspace import CloudWorkspace

## Configuration credential

In [5]:
# Configuration for the credential
import os
from dotenv import load_dotenv

ENV_PATH = ".env"
load_dotenv(ENV_PATH)

False

## Create project on evidently AI

In [6]:
ws = CloudWorkspace(token=os.getenv("EVIDENTLY_TOKEN"), url="https://app.evidently.cloud")

## Create a Project within your Organization, or connect to an existing Project:

In [7]:
try:
    existing_projects = ws.list_projects()
    project = next((p for p in existing_projects if p.name == "LLM Audit"), None)
    
    if project is None:
        project = ws.create_project(name="LLM Audit", org_id="019d6885-06c6-7601-9a3e-2298bed991f4")
        project.description = "Project for auditing LLM performance on models, data, and LLMs evaluations."
        project.save()
        print(f"✅ Project created: {project.name}")   # ← print, not project()
    else:
        print(f"📁 Project already exists: {project.name}")  # ← print, not project()

except Exception as e:
    print(f"Error accessing or creating project: {e}")

print(f"Project name: {project.name}")

📁 Project already exists: LLM Audit
Project name: LLM Audit


## Prepare the dataset -> Merge prompt with dataframe into reliable data information
* ### Load dataset
* ### Create prompt

## Load dataset

In [8]:
pd.set_option('display.max_columns', None)
df = pd.read_parquet("../../Database/data/eda_banking.parquet")
df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Complain,Satisfaction Score,Card Type,Point Earned,Fraud,RiskScore,BalancePerProduct,AgeRisk,HighValueCustomer,LowCreditRisk,MarketingScore,ComplainFlag,LowSatisfaction,OperationalRiskScore
0,619,France,Female,42,2,11983969.0,1,1,1,10134888.0,1,1,2,DIAMOND,464,0,1,0.000000,0,0,0,0,1,1,2
1,608,Spain,Female,41,1,8380786.0,1,0,1,11254258.0,0,1,3,DIAMOND,456,0,1,41903.930000,0,0,0,1,1,0,1
2,502,France,Female,42,8,15966080.0,3,1,0,11393157.0,1,1,3,DIAMOND,377,0,3,39915.200000,0,1,0,1,1,0,2
3,699,France,Female,39,1,11983969.0,2,0,0,9382663.0,0,0,5,GOLD,350,0,1,0.000000,0,0,0,0,0,0,1
4,850,Spain,Female,43,2,12551082.0,1,1,1,7908410.0,0,0,5,GOLD,425,0,1,62755.410000,0,1,0,2,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,Male,39,5,11983969.0,2,1,0,9627064.0,0,0,1,DIAMOND,300,0,1,0.000000,0,0,0,0,0,1,2
9996,516,France,Male,35,10,5736961.0,1,1,1,10169977.0,0,0,5,PLATINUM,771,0,0,28684.805000,0,0,0,0,0,0,0
9997,709,France,Female,36,7,11983969.0,1,0,1,4208558.0,1,1,3,SILVER,564,0,1,0.000000,0,0,0,0,1,0,1
9998,772,Germany,Male,42,3,7507531.0,2,1,0,9288852.0,1,1,2,GOLD,339,1,2,25025.103333,0,0,0,1,1,1,3


## Create LLMs prompt for evidentlyAI production

In [9]:
# Define questions and context
question_and_context = [
    [
        "What is the fraud detection rate in the dataset?",
        "Fraud detection in banking uses ML classifiers. Precision-recall tradeoffs are critical."
    ],
    [
        "Which ML model performs best for credit default prediction?",
        "XGBoost and Random Forest outperform logistic regression on imbalanced datasets."
    ],
    [
        "How should missing values be handled in operational banking data?",
        "Imputation must preserve regulatory compliance and not introduce synthetic bias."
    ],
    [
        "What is the churn rate and how can it be reduced?",
        "Customer churn in retail banking averages 15-20% annually."
    ],
    [
        "Explain the class imbalance issue in fraud detection.",
        "Fraud datasets are heavily imbalanced with fraud rates below 1-2%."
    ],

    [
        "What are the key performance metrics for evaluating fraud detection models?",
        "Key metrics include precision, recall, F1-score, and AUC-ROC."
    ],

    [
        "How does good corporate governance influence fraud prevention in banking sector?",
        "Examine the effect of good corporate governance on fraud prevention in the banking sector, including the role of internal controls, risk management practices, and ethical standards in mitigating fraudulent activities."
    ],

    [
        "What's advantage MLOps environment for banking industry?",
        "MLOps environments enable continuous integration and deployment of ML/DL models, ensuring faster iteration and improved model performance in the banking industry. the effectiveness of MLOps enviroment in the banking industry, including its impact on model deployment speed, scalability, and overall performance improvement to meet the dynamic needs of the banking sector."
    ]
]

pd.set_option('display.max_colwidth', None)
eval_df = pd.DataFrame(question_and_context, columns=["question", "context"])

In [10]:
eval_df

,question,context
0,What is the fraud detection rate in the dataset?,Fraud detection in banking uses ML classifiers. Precision-recall tradeoffs are critical.
1,Which ML model performs best for credit default prediction?,XGBoost and Random Forest outperform logistic regression on imbalanced datasets.
2,How should missing values be handled in operational banking data?,Imputation must preserve regulatory compliance and not introduce synthetic bias.
3,What is the churn rate and how can it be reduced?,Customer churn in retail banking averages 15-20% annually.
4,Explain the class imbalance issue in fraud detection.,Fraud datasets are heavily imbalanced with fraud rates below 1-2%.
5,What are the key performance metrics for evaluating fraud detection models?,"Key metrics include precision, recall, F1-score, and AUC-ROC."
6,How does good corporate governance influence fraud prevention in banking sector?,"Examine the effect of good corporate governance on fraud prevention in the banking sector, including the role of internal controls, risk management practices, and ethical standards in mitigating fraudulent activities."
7,What's advantage MLOps environment for banking industry?,"MLOps environments enable continuous integration and deployment of ML/DL models, ensuring faster iteration and improved model performance in the banking industry. the effectiveness of MLOps enviroment in the banking industry, including its impact on model deployment speed, scalability, and overall performance improvement to meet the dynamic needs of the banking sector."


In [11]:
from groq import Groq
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

client = Groq(api_key=os.environ.get("GROQ_API_KEY_2"))

# 1. Build data context from dataframe
def build_data_context(df):
    context = f"""
You are a banking data analyst & data scentist. Use the following dataset information to reasong and answer questions.

DATASET OVERVIEW:
- Total Records: {len(df):,}
- Columns: {list(df.columns)}

STATISTICAL SUMMARY:
{df.describe().to_string()}

CATEGORICAL COLUMNS:
- Geography Distribution: {df['Geography'].value_counts().to_dict()}
- Gender Distribution: {df['Gender'].value_counts().to_dict()}
- Card Type Distribution: {df['Card Type'].value_counts().to_dict()}

- KEY METRICS:
- Fraud Rate: {df['Fraud'].mean():.2%}
- Churn Rate (Exited): {df['Exited'].mean():.2%}
- Avg Credit Score: {df['CreditScore'].mean():.2f}
- Avg Balance : {df['Balance'].mean():.2f}
- Avg Risk Score: {df['RiskScore'].mean():.2f}
- High Value Customers: {df['HighValueCustomer'].sum():,} ({df['HighValueCustomer'].mean():.2%})
- Customers with Complaints: {df['Complain'].sum():,} ({df['Complain'].mean():.2%})
- Active Members: {df['IsActiveMember'].sum():,} ({df['IsActiveMember'].mean():.2%})

CORRELATION WITH FRAUD (top features):
{df.corr(numeric_only=True)['Fraud'].sort_values(ascending=False).to_string()}

CORRELATION WITH CHURN (top features):
{df.corr(numeric_only=True)['Exited'].sort_values(ascending=False).to_string()}
"""
    
    return context

data_context = build_data_context(df)


In [12]:

# 2. Generate answers with data-aware prompting
def get_groq_response_with_context(row):
    prompt = f"""
{data_context}

---
CONTEXT HINT: {row['context']}
QUESTION: {row['question']}

Answer based strictly on the dataset statistics provided above. Be specific with numbers and percentages from the data.
"""
    
    completion = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
        max_tokens=1024,
        top_p=0.9,
        stream=False
    )
    return completion.choices[0].message.content

In [13]:
# apply to eval_df
eval_df['answer'] = eval_df.apply(get_groq_response_with_context, axis=1)
eval_df

,question,context,answer
0,What is the fraud detection rate in the dataset?,Fraud detection in banking uses ML classifiers. Precision-recall tradeoffs are critical.,"To determine the fraud detection rate in the dataset, we need to refer to the 'Fraud Rate' metric provided in the KEY METRICS section. \n\nThe Fraud Rate is given as 3.32%. This means that out of the total 10,000 records, 3.32% of the customers are identified as fraudulent. \n\nTo calculate the actual number of fraudulent customers, we can multiply the total number of records by the fraud rate:\n\n10,000 * 0.0332 = 332\n\nSo, there are approximately 332 fraudulent customers in the dataset.\n\nTherefore, the fraud detection rate in the dataset is 3.32% or approximately 332 out of 10,000 customers."
1,Which ML model performs best for credit default prediction?,XGBoost and Random Forest outperform logistic regression on imbalanced datasets.,"To determine the best ML model for credit default prediction, we need to consider the characteristics of the dataset and the performance metrics of different models. \n\nGiven that the dataset is imbalanced (Fraud Rate: 3.32%), we can expect that models that handle imbalanced datasets well will perform better. The context hint suggests that XGBoost and Random Forest outperform logistic regression on imbalanced datasets.\n\nAlthough the dataset statistics do not provide direct performance metrics for XGBoost, Random Forest, and logistic regression, we can infer the potential performance based on the characteristics of the dataset.\n\nThe top features correlated with Fraud are:\n1. Exited (0.292838)\n2. ComplainFlag (0.292243)\n3. Complain (0.292243)\n4. OperationalRiskScore (0.283743)\n5. RiskScore (0.223436)\n\nThese features are also correlated with Churn (Exited), which has a rate of 20.38%. The correlation between Fraud and Churn suggests that models that can handle multiple related targets (e.g., multi-task learning) might perform well.\n\nConsidering the imbalanced nature of the dataset and the correlation between features, XGBoost is likely to perform well. XGBoost is an ensemble model that can handle imbalanced datasets and is robust to outliers. It can also handle multiple features and interactions between them, which is beneficial given the correlations between features in this dataset.\n\nRandom Forest is another ensemble model that can handle imbalanced datasets and is robust to overfitting. However, it might not perform as well as XGBoost, especially when dealing with complex interactions between features.\n\nLogistic regression, on the other hand, is a linear model that might not perform well on imbalanced datasets, especially when compared to ensemble models like XGBoost and Random Forest.\n\nBased on the dataset statistics and the context hint, XGBoost is likely to perform best for credit default prediction, followed by Random Forest, and then logistic regression.\n\nTo quantify the potential performance, we can consider the Fraud Rate (3.32%) and the correlation between features. A good model should be able to predict at least 80-90% of the non-fraudulent cases correctly (i.e., specificity) while maintaining a reasonable sensitivity (e.g., 50-60% of fraudulent cases correctly predicted).\n\nIn terms of specific numbers, a well-performing XGBoost model might achieve:\n- Accuracy: 96-97% (given the low Fraud Rate)\n- Precision: 80-90% (depending on the threshold)\n- Recall (Sensitivity): 50-60% (depending on the threshold)\n- F1-score: 0.6-0.7 (depending on the threshold)\n\nNote that these numbers are estimates based on the dataset statistics and may vary depending on the actual model performance and the chosen threshold."
2,How should missing values be handled in operational banking data?,Imputation must preserve regulatory compliance and not introduce synthetic bias.,"To handle missing values in operational banking data while preserving regulatory compliance and avoiding synthetic bias, we should first examine th

In [14]:
# Fillna missing values
eval_df['answer'] = eval_df['answer'].fillna("No answer generated.")

## Auditing the LLMs for evaluation

* ###  Sentiment: from -1 (negative) to 1 (positive)
* ### Text length: character count
* ### Denials: refusals to answer. This uses an LLM-as-a-judge with built-in prompt.

### Each evaluation is a descriptor. It adds a new score or label to each row in your dataset. For LLM-as-a-judge, we’ll use OpenAI GPT-4o mini. Set OpenAI key as an environment variable:

## Feature engineering LLMs

In [15]:
import re

# Length (already handled by descriptor, but we keep explicit)
eval_df["answer_length"] = eval_df["answer"].apply(len)

# Word count
eval_df["word_count"] = eval_df["answer"].apply(lambda x: len(x.split()))

# Contains number (important for banking correctness)
eval_df["contains_number"] = eval_df["answer"].str.contains(
    r"i cannot|i can't|i do not|i don't know|cannot assist",
    case=False,
    regex=True
)

# Uncertainty detection (hallucination proxy signal)
eval_df["uncertain"] = eval_df["answer"].str.contains(
    r"maybe|possibly|might|approximately|around",
    case=False,
    regex=True
)

# Numeric density (how data-driven the answer is)
def numeric_density(text):
    nums = re.findall(r"\d+", text)
    words = text.split()
    return len(nums) / max(len(words), 1)

eval_df["numeric_density"] = eval_df["answer"].apply(numeric_density)

# Check if answer include percentages
eval_df["contains_percentage"] = eval_df["answer"].str.contains("%")

# fraud mentions
eval_df["mentions_fraud"] = eval_df["answer"].str.contains("fraud", case=False)

# churn mentions
eval_df["mentions_churn"] = eval_df["answer"].str.contains("churn", case=False)

In [16]:
# Turn on debug
import litellm

litellm._turn_on_debug()

os.environ["GROQ_API_KEY_2"] = os.environ.get("GROQ_API_KEY_2", "")

response = litellm.completion(
    model="groq/llama-3.1-8b-instant",
    messages=[{"role": "user", "content": "Say OK"}],
)
print(response.choices[0].message.content)

22:27:09 - LiteLLM:DEBUG: utils.py:481 - 

22:27:09 - LiteLLM:DEBUG: utils.py:481 - Request to litellm:
22:27:09 - LiteLLM:DEBUG: utils.py:481 - litellm.completion(model='groq/llama-3.1-8b-instant', messages=[{'role': 'user', 'content': 'Say OK'}])
22:27:09 - LiteLLM:DEBUG: utils.py:481 - 

22:27:09 - LiteLLM:DEBUG: utils.py:1007 - Removing thought signatures from tool call IDs for non-Gemini model
22:27:09 - LiteLLM:DEBUG: litellm_logging.py:557 - self.optional_params: {}
22:27:09 - LiteLLM:DEBUG: utils.py:481 - SYNC kwargs[caching]: False; litellm.cache: None; kwargs.get('cache')['no-cache']: False
22:27:09 - LiteLLM:DEBUG: utils.py:5605 - checking potential_model_names in litellm.model_cost: {'split_model': 'llama-3.1-8b-instant', 'combined_model_name': 'groq/llama-3.1-8b-instant', 'stripped_model_name': 'llama-3.1-8b-instant', 'combined_stripped_model_name': 'groq/llama-3.1-8b-instant', 'custom_llm_provider': 'groq'}
22:27:09 - LiteLLM:DEBUG: utils.py:5605 - checking potential_mode

OK


22:27:10 - LiteLLM:DEBUG: litellm_logging.py:1540 - response_cost: 2.01e-06
22:27:10 - LiteLLM:DEBUG: utils.py:5605 - checking potential_model_names in litellm.model_cost: {'split_model': 'llama-3.1-8b-instant', 'combined_model_name': 'groq/llama-3.1-8b-instant', 'stripped_model_name': 'llama-3.1-8b-instant', 'combined_stripped_model_name': 'groq/llama-3.1-8b-instant', 'custom_llm_provider': 'groq'}
22:27:10 - LiteLLM:DEBUG: utils.py:5605 - checking potential_model_names in litellm.model_cost: {'split_model': 'llama-3.1-8b-instant', 'combined_model_name': 'groq/llama-3.1-8b-instant', 'stripped_model_name': 'llama-3.1-8b-instant', 'combined_stripped_model_name': 'groq/llama-3.1-8b-instant', 'custom_llm_provider': 'groq'}


In [17]:
import litellm
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

os.environ["GROQ_API_KEY"] = os.environ.get("GROQ_API_KEY_2", "")
print(f"Key loaded: {bool(os.environ.get('GROQ_API_KEY_2'))}")

# ── Option 1: Tell LiteLLM to auto-retry with backoff ─────────────────────
litellm.num_retries = 5
litellm.retry_after = 10  # wait 10s between retries

eval_dataset = Dataset.from_pandas(
    eval_df,
    data_definition=DataDefinition(),
    descriptors=[
        Sentiment("answer"),
        TextLength("answer"),
    ]
)

eval_dataset.as_dataframe()

Key loaded: True


,question,context,answer,answer_length,word_count,contains_number,uncertain,numeric_density,contains_percentage,mentions_fraud,mentions_churn,Sentiment for answer,text_length
0,What is the fraud detection rate in the dataset?,Fraud detection in banking uses ML classifiers. Precision-recall tradeoffs are critical.,"To determine the fraud detection rate in the dataset, we need to refer to the 'Fraud Rate' metric provided in the KEY METRICS section. \n\nThe Fraud Rate is given as 3.32%. This means that out of the total 10,000 records, 3.32% of the customers are identified as fraudulent. \n\nTo calculate the actual number of fraudulent customers, we can multiply the total number of records by the fraud rate:\n\n10,000 * 0.0332 = 332\n\nSo, there are approximately 332 fraudulent customers in the dataset.\n\nTherefore, the fraud detection rate in the dataset is 3.32% or approximately 332 out of 10,000 customers.",593,100,False,True,0.170000,True,True,False,-0.9818,593
1,Which ML model performs best for credit default prediction?,XGBoost and Random Forest outperform logistic regression on imbalanced datasets.,"To determine the best ML model for credit default prediction, we need to consider the characteristics of the dataset and the performance metrics of different models. \n\nGiven that the dataset is imbalanced (Fraud Rate: 3.32%), we can expect that models that handle imbalanced datasets well will perform better. The context hint suggests that XGBoost and Random Forest outperform logistic regression on imbalanced datasets.\n\nAlthough the dataset statistics do not provide direct performance metrics for XGBoost, Random Forest, and logistic regression, we can infer the potential performance based on the characteristics of the dataset.\n\nThe top features correlated with Fraud are:\n1. Exited (0.292838)\n2. ComplainFlag (0.292243)\n3. Complain (0.292243)\n4. OperationalRiskScore (0.283743)\n5. RiskScore (0.223436)\n\nThese features are also correlated with Churn (Exited), which has a rate of 20.38%. The correlation between Fraud and Churn suggests that models that can handle multiple related targets (e.g., multi-task learning) might perform well.\n\nConsidering the imbalanced nature of the dataset and the correlation between features, XGBoost is likely to perform well. XGBoost is an ensemble model that can handle imbalanced datasets and is robust to outliers. It can also handle multiple features and interactions between them, which is beneficial given the correlations between features in this dataset.\n\nRandom Forest is another ensemble model that can handle imbalanced datasets and is robust to overfitting. However, it might not perform as well as XGBoost, especially when dealing with complex interactions between features.\n\nLogistic regression, on the other hand, is a linear model that might not perform well on imbalanced datasets, especially when compared to ensemble models like XGBoost and Random Forest.\n\nBased on the dataset statistics and the context hint, XGBoost is likely to perform best for credit default prediction, followed by Random Forest, and then logistic regression.\n\nTo quantify the potential performance, we can consider the Fraud Rate (3.32%) and the correlation between features. A good model should be able to predict at least 80-90% of the non-fraudulent cases correctly (i.e., specificity) while maintaining a reasonable sensitivity (e.g., 50-60% of fraudulent cases correctly predicted).\n\nIn terms of specific numbers, a well-performing XGBoost model might achieve:\n- Accuracy: 96-97% (given the low Fraud Rate)\n- Precision: 80-90% (depending on the threshold)\n- Recall (Sensitivity): 50-60% (depending on the threshold)\n- F1-score: 0.6-0.7 (depending on the threshold)\n\nNote that these numbers are estimates based on the dataset statistics and may vary depending on the actual model performance and the chosen threshold.",2755,405,False,True,0.088889,True,True,True,0.9524,2755
2,How should missing values be hand

## Deploy to evidentlyAI local UI Dashboard

In [18]:
from evidently.presets import DataSummaryPreset
report = Report(
    metrics=[
        DataSummaryPreset()
    ],
    tags=["LLM-Audit", "groq", "Banking"]
)
snapshot = report.run(current_data=eval_dataset)
print("✅ Report generated successfully")

# Upload to evidently cloud
ws.add_run(project.id, snapshot)
print(f"LLMs is deployed to evidently cloud successfully")

✅ Report generated successfully
LLMs is deployed to evidently cloud successfully


In [19]:
eval_df

,question,context,answer,answer_length,word_count,contains_number,uncertain,numeric_density,contains_percentage,mentions_fraud,mentions_churn
0,What is the fraud detection rate in the dataset?,Fraud detection in banking uses ML classifiers. Precision-recall tradeoffs are critical.,"To determine the fraud detection rate in the dataset, we need to refer to the 'Fraud Rate' metric provided in the KEY METRICS section. \n\nThe Fraud Rate is given as 3.32%. This means that out of the total 10,000 records, 3.32% of the customers are identified as fraudulent. \n\nTo calculate the actual number of fraudulent customers, we can multiply the total number of records by the fraud rate:\n\n10,000 * 0.0332 = 332\n\nSo, there are approximately 332 fraudulent customers in the dataset.\n\nTherefore, the fraud detection rate in the dataset is 3.32% or approximately 332 out of 10,000 customers.",593,100,False,True,0.170000,True,True,False
1,Which ML model performs best for credit default prediction?,XGBoost and Random Forest outperform logistic regression on imbalanced datasets.,"To determine the best ML model for credit default prediction, we need to consider the characteristics of the dataset and the performance metrics of different models. \n\nGiven that the dataset is imbalanced (Fraud Rate: 3.32%), we can expect that models that handle imbalanced datasets well will perform better. The context hint suggests that XGBoost and Random Forest outperform logistic regression on imbalanced datasets.\n\nAlthough the dataset statistics do not provide direct performance metrics for XGBoost, Random Forest, and logistic regression, we can infer the potential performance based on the characteristics of the dataset.\n\nThe top features correlated with Fraud are:\n1. Exited (0.292838)\n2. ComplainFlag (0.292243)\n3. Complain (0.292243)\n4. OperationalRiskScore (0.283743)\n5. RiskScore (0.223436)\n\nThese features are also correlated with Churn (Exited), which has a rate of 20.38%. The correlation between Fraud and Churn suggests that models that can handle multiple related targets (e.g., multi-task learning) might perform well.\n\nConsidering the imbalanced nature of the dataset and the correlation between features, XGBoost is likely to perform well. XGBoost is an ensemble model that can handle imbalanced datasets and is robust to outliers. It can also handle multiple features and interactions between them, which is beneficial given the correlations between features in this dataset.\n\nRandom Forest is another ensemble model that can handle imbalanced datasets and is robust to overfitting. However, it might not perform as well as XGBoost, especially when dealing with complex interactions between features.\n\nLogistic regression, on the other hand, is a linear model that might not perform well on imbalanced datasets, especially when compared to ensemble models like XGBoost and Random Forest.\n\nBased on the dataset statistics and the context hint, XGBoost is likely to perform best for credit default prediction, followed by Random Forest, and then logistic regression.\n\nTo quantify the potential performance, we can consider the Fraud Rate (3.32%) and the correlation between features. A good model should be able to predict at least 80-90% of the non-fraudulent cases correctly (i.e., specificity) while maintaining a reasonable sensitivity (e.g., 50-60% of fraudulent cases correctly predicted).\n\nIn terms of specific numbers, a well-performing XGBoost model might achieve:\n- Accuracy: 96-97% (given the low Fraud Rate)\n- Precision: 80-90% (depending on the threshold)\n- Recall (Sensitivity): 50-60% (depending on the threshold)\n- F1-score: 0.6-0.7 (depending on the threshold)\n\nNote that these numbers are estimates based on the dataset statistics and may vary depending on the actual model performance and the chosen threshold.",2755,405,False,True,0.088889,True,True,True
2,How should missing values be handled in operational banking data?,Imputation must preserve